# Dataset C: Basketball-Reference Box Plus/Minus (2015-16 to 2025-26)

This notebook builds the **Box +/-** features from `ListOfPotentialFeatures.md`:

| Feature | Column | Level |
|---|---|---|
| Regular-season Box +/- | `rs_bpm` | player-season |
| Playoff Box +/- | `po_bpm` | player-season |
| 3-year trend: Box +/- | `avg_bpm_last_3` (the three seasons before this one, regular season, same window as notebook 01, weighted by minutes) | player-season |

**Box Plus/Minus (BPM)** is Basketball-Reference's box-score estimate of a player's points per 100 possessions above a league-average player. 0 is average, +5 is All-Star level, and −2 is roughly replacement level. Unlike on-court plus/minus (notebooks 01-02), it is built from the player's own box score, so it depends much less on his teammates. llimllib's database doesn't have it, so it comes from Basketball-Reference.

*Run order: `01` → `02` → `03` → `999`.*

- **Dataset name:** Basketball-Reference "Advanced" player stats, one page per season and season type
- **Links:** `https://www.basketball-reference.com/leagues/NBA_{end year}_advanced.html` (regular season) and `https://www.basketball-reference.com/playoffs/NBA_{end year}_advanced.html` (playoffs)
- **Pages used:** regular season 2012-13 through 2025-26 (the three extra seasons are for the 3-year average) and playoffs 2015-16 through 2025-26, 25 pages in total. Basketball-Reference asks scripts to make **at most 20 requests per minute**, so the download waits 4 seconds between pages. That takes under 2 minutes the first time and is skipped after.
- **Saved as:** `data-files/00-raw/bbref_advanced_html/{regular_season|playoffs}_{end year}.html` (each page exactly as downloaded)
- **Also reads:** `data-files/00-raw/llimllib_bbref_salaries_raw.csv` (downloaded here if missing, the same file `salary-data/02` uses) to translate Basketball-Reference player ids into NBA ids

## Important variables

| Variable | Units | Meaning |
|---|---|---|
| `player` + its `data-append-csv` attribute | text | Player name and Basketball-Reference player id (e.g. `jokicni01`). |
| `team_id` | text | Team abbreviation. A player traded mid-season has one row per team **plus** a combined row (`TOT`, or `2TM` / `3TM` in recent seasons). |
| `g`, `mp` | games, minutes | Games and total minutes. |
| `bpm` (`obpm`, `dbpm`) | points per 100 possessions | Box Plus/Minus (offensive and defensive parts). |

## Concerns going in

1. **Traded players have several rows.** We keep only the combined season row, to match notebook 01's one row per player-season.
2. **Basketball-Reference uses its own player ids.** We translate them to NBA ids through llimllib's `bbref_salaries` table, which lists both ids. Players it doesn't cover are matched by name within the season, but only when the name is unique.
3. **BPM is unstable for tiny samples.** A player with a few minutes can have an extreme BPM, positive or negative. We keep the single-season values, as with the other per-game stats; `rs_mpg` / `rs_gp` show the sample size. But an extreme value would distort a plain average, so the 3-year average is weighted by minutes (section 4C).

## Setup

In [1]:
import re
import sys
import time
from pathlib import Path

import bs4
import duckdb
import numpy as np
import pandas as pd
import requests

sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import RAW, PROCESSED, FIRST_SEASON, LAST_SEASON, normalize_name, prior_seasons_average

HISTORY_FIRST = FIRST_SEASON - 3

## 1. Download (skipped for pages that already exist)

In [2]:
HTML_DIR = RAW / 'bbref_advanced_html'
HTML_DIR.mkdir(exist_ok=True)
URLS = {'regular_season': 'https://www.basketball-reference.com/leagues/NBA_{}_advanced.html',
        'playoffs': 'https://www.basketball-reference.com/playoffs/NBA_{}_advanced.html'}
pages = ([('regular_season', s) for s in range(HISTORY_FIRST, LAST_SEASON + 1)]
         + [('playoffs', s) for s in range(FIRST_SEASON, LAST_SEASON + 1)])

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36'}
downloaded = 0
for season_type, season in pages:
    path = HTML_DIR / f'{season_type}_{season}.html'
    if path.exists():
        continue
    if downloaded:
        time.sleep(4)                      # stay under 20 requests per minute
    r = requests.get(URLS[season_type].format(season), headers=headers, timeout=60)
    r.raise_for_status()
    path.write_bytes(r.content)
    downloaded += 1
print('pages downloaded now:', downloaded, '| pages on disk:', len(list(HTML_DIR.glob('*.html'))), 'of', len(pages))

RAW_IDS = RAW / 'llimllib_bbref_salaries_raw.csv'
if not RAW_IDS.exists():
    con = duckdb.connect()
    con.sql('INSTALL httpfs; LOAD httpfs;')
    con.sql("ATTACH 'https://basketball-data.billmill.org/nba/nba.duckdb' AS nba (READ_ONLY)")
    con.sql('SELECT * FROM nba.bbref_salaries ORDER BY season, team_id, bbref_id').df().to_csv(RAW_IDS, index=False)
    print('downloaded', RAW_IDS.name)

pages downloaded now: 0 | pages on disk: 25 of 25


## 2. Load the data

Each page has one table of player rows. Every cell carries a `data-stat` attribute naming its column, so we read the cells by that name. The table repeats its header row every 20 players; those rows have the class `thead` and are skipped.

The two kinds of page use **two layouts**. The regular-season pages use a newer one (table id `advanced`, columns `name_display`, `team_name_abbr`, `games`), and the playoff pages an older one (table id `advanced_stats`, columns `player`, `team_id`, `g`). The statistics are the same, so we rename the newer columns to the older names.

In [3]:
def read_advanced(path):
    soup = bs4.BeautifulSoup(path.read_text(encoding='utf-8'), 'html.parser')
    table = soup.find('table', id='advanced_stats') or soup.find('table', id='advanced')
    rows = []
    for tr in table.find('tbody').find_all('tr'):
        if 'thead' in (tr.get('class') or []):
            continue
        cells = {c.get('data-stat'): c.get_text(strip=True) for c in tr.find_all(['th', 'td'])}
        player_cell = tr.find(attrs={'data-stat': re.compile('^(player|name_display)$')})
        cells['bbref_id'] = player_cell.get('data-append-csv') if player_cell else None
        rows.append(cells)
    # The regular-season pages use a newer layout with different column names; use one set of names.
    return pd.DataFrame(rows).rename(columns={'name_display': 'player', 'team_name_abbr': 'team_id', 'games': 'g'})

frames = []
for season_type, season in pages:
    df = read_advanced(HTML_DIR / f'{season_type}_{season}.html')
    frames.append(df.assign(season_type=season_type, season=season))
bbref_raw = pd.concat(frames, ignore_index=True)
print('rows:', len(bbref_raw), '| columns:', bbref_raw.shape[1])
bbref_raw[['season_type', 'season', 'player', 'bbref_id', 'team_id', 'g', 'mp', 'obpm', 'dbpm', 'bpm']].head()

rows: 11849 | columns: 33


,season_type,season,player,bbref_id,team_id,g,mp,obpm,dbpm,bpm
0,regular_season,2013,Damian Lillard,lillada01,POR,82,3167,2.2,-1.9,0.3
1,regular_season,2013,Kevin Durant,duranke01,OKC,81,3119,7.6,1.7,9.2
2,regular_season,2013,Monta Ellis,ellismo01,MIL,82,3076,0.3,0.3,0.6
3,regular_season,2013,DeMar DeRozan,derozde01,TOR,82,3013,-0.2,-1.2,-1.4
4,regular_season,2013,Kobe Bryant,bryanko01,LAL,78,3013,4.7,-0.2,4.6


## 3A. Is the data tidy?

Each row should be one player, team, and season. A traded player should have one row per team **and** one combined row. Rows with no player id (a "League Average" summary row, if any) are not players.

In [4]:
print('rows with no bbref_id:', bbref_raw['bbref_id'].isna().sum(), bbref_raw.loc[bbref_raw['bbref_id'].isna(), 'player'].unique()[:5])
bb = bbref_raw[bbref_raw['bbref_id'].notna()].copy()

key = ['season_type', 'season', 'bbref_id']
bb['is_combined'] = bb['team_id'].str.fullmatch(r'TOT|\dTM')
bb['rows_for_player'] = bb.groupby(key)['team_id'].transform('size')
multi = bb[bb['rows_for_player'] > 1]
print('player-seasons with more than one row:', multi.groupby(key).ngroups)
print('...each has exactly one combined row:', multi.groupby(key)['is_combined'].sum().eq(1).all())
print('combined-row labels used:', sorted(bb.loc[bb['is_combined'], 'team_id'].unique()))
print('combined rows for players with only one row:', (bb['is_combined'] & (bb['rows_for_player'] == 1)).sum())

bb[(bb['player'] == 'Luka Dončić') & (bb['season'] == 2025) & (bb['season_type'] == 'regular_season')][['player', 'team_id', 'g', 'mp', 'bpm']]

rows with no bbref_id: 14 <StringArray>
['League Average']
Length: 1, dtype: str
player-seasons with more than one row: 974
...each has exactly one combined row: True
combined-row labels used: ['2TM', '3TM', '4TM']
combined rows for players with only one row: 0


,player,team_id,g,mp,bpm
8124,Luka Dončić,2TM,50,1769,6.7
8125,Luka Dončić,DAL,22,785,6.9
8126,Luka Dončić,LAL,28,984,6.5


In [5]:
bb = bb[(bb['rows_for_player'] == 1) | bb['is_combined']].copy()
for col in ['g', 'mp', 'obpm', 'dbpm', 'bpm']:
    bb[col] = pd.to_numeric(bb[col], errors='coerce')
print('one row per season type + season + player now:', not bb.duplicated(key).any())

one row per season type + season + player now: True


**What we found:** Every page parsed into a table with one column per `data-stat`. Each regular-season page ends with a **"League Average" row** (14 in total), which has no player id and is dropped. **A traded player has one row per team plus one combined row**, labeled `2TM`, `3TM`, or `4TM` by the number of teams. Dončić's 2024-25 season has a DAL row (22 games), a LAL row (28 games), and a `2TM` row with all 50. All 974 multi-row player-seasons have exactly one combined row, so keeping it (and dropping the per-team rows) gives one row per player-season, matching notebook 01. After this, the table is tidy.

## 3B. Size of the data

In [6]:
bb.groupby(['season', 'season_type']).size().unstack().T

season,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
season_type,,,,,,,,,,,,,,
playoffs,NaN,NaN,NaN,215.0,215.0,210.0,212.0,217.0,239.0,217.0,217.0,214.0,219.0,230.0
regular_season,469.0,482.0,492.0,476.0,486.0,540.0,530.0,529.0,540.0,605.0,539.0,572.0,569.0,582.0


**The counts are identical to llimllib's in every season:** 469-605 regular-season players per season (notebook 01) and 210-239 playoff players (notebook 02). Sections 4A-4B check the match row by row.

## 3C. Missing data

In [7]:
bb[['player', 'team_id', 'g', 'mp', 'obpm', 'dbpm', 'bpm']].isna().sum()

player     0
team_id    0
g          0
mp         0
obpm       1
dbpm       1
bpm        1
dtype: int64

In [8]:
blank_bpm = bb[bb['bpm'].isna()]
print('rows with blank bpm:', len(blank_bpm), '| their minutes:', blank_bpm['mp'].describe()[['min', 'max']].to_dict() if len(blank_bpm) else '-')

rows with blank bpm: 1 | their minutes: {'min': 0.0, 'max': 0.0}


**What we found:** Only 1 row has a blank `bpm`, a player credited with a game but 0 minutes, where BPM can't be computed. It keeps a blank `bpm`. Every other column is complete.

## 3D. Outliers and suspicious entries

In [9]:
print(bb.groupby('season_type')['bpm'].describe().round(1))
small = bb['mp'] < 50
print('\nBPM range, players with under 50 minutes:', bb.loc[small, 'bpm'].min(), 'to', bb.loc[small, 'bpm'].max())
print('BPM range, players with 500+ minutes:  ', bb.loc[bb['mp'] >= 500, 'bpm'].min(), 'to', bb.loc[bb['mp'] >= 500, 'bpm'].max())

                 count  mean  std   min  25%  50%  75%    max
season_type                                                  
playoffs        2405.0  -0.6  8.2 -58.2 -3.5 -0.1  2.8   70.1
regular_season  7410.0  -1.6  5.5 -54.4 -3.3 -1.3  0.6  240.8

BPM range, players with under 50 minutes: -58.2 to 240.8
BPM range, players with 500+ minutes:   -9.6 to 14.2


In [10]:
# Highest regular-season BPM each season among players with 1,500+ minutes
(bb[(bb['season_type'] == 'regular_season') & (bb['mp'] >= 1500) & (bb['season'] >= FIRST_SEASON)]
   .sort_values('bpm', ascending=False).groupby('season').head(1).sort_values('season')[['season', 'player', 'mp', 'bpm']])

,season,player,mp,bpm
1859,2016,Stephen Curry,2700,11.9
2424,2017,Russell Westbrook,2802,11.0
3043,2018,James Harden,2551,9.8
3679,2019,James Harden,2867,10.9
4457,2020,Giannis Antetokounmpo,1917,11.4
5041,2021,Nikola Jokić,2488,12.0
5766,2022,Nikola Jokić,2476,13.6
6611,2023,Nikola Jokić,2323,12.9
7246,2024,Nikola Jokić,2737,13.1
7997,2025,Nikola Jokić,2571,13.3


**What we found:**

- **The extreme BPMs all come from tiny samples.** Players with under 50 minutes range from −58.2 to +240.8, while players with 500+ minutes stay between −9.6 and +14.2. That is a property of the formula (a few possessions decide the number), not a data error. We keep the single-season values, since `rs_mpg` and `rs_gp` let the model discount them, and weight the 3-year average by minutes.
- **The league leaders look right.** The top BPM among high-minute players each season is the player you would expect: Stephen Curry (11.9 in 2015-16), Russell Westbrook (2016-17), James Harden (2017-18 and 2018-19), Giannis Antetokounmpo (2019-20), and Nikola Jokić every season from 2020-21 on (12.0 to 14.2).

## 4. Clean the data

### 4A. Translate Basketball-Reference ids to NBA ids

1. **By id:** llimllib's `bbref_salaries` lists each player's Basketball-Reference id next to his NBA id. Each Basketball-Reference id maps to one NBA id.
2. **By name (fallback):** for ids not in that table, match `season` + `name_key` against llimllib's player rows from notebooks 01-02, **only if** the name is unique in that season on both sides.

In [11]:
ids_raw = pd.read_csv(RAW_IDS, dtype={'player_id': str, 'bbref_id': str})
id_map = ids_raw.dropna(subset=['player_id']).drop_duplicates(['bbref_id', 'player_id'])
assert not id_map['bbref_id'].duplicated().any(), 'a bbref id maps to two NBA ids'
id_map = id_map.set_index('bbref_id')['player_id']

bb['nba_player_id'] = bb['bbref_id'].map(id_map)
bb['match'] = np.where(bb['nba_player_id'].notna(), 'bbref id', None)

# Fallback: season + unique name, against llimllib's player-seasons (all seasons we load)
stats_raw = pd.read_csv(RAW / 'llimllib_player_season_stats_raw.csv', dtype={'player_id': str})
players = pd.read_csv(RAW / 'llimllib_players_raw.csv', dtype={'player_id': str})
nba_names = (stats_raw[['season', 'player_id']].drop_duplicates()
             .merge(players[['player_id', 'name']], on='player_id'))
nba_names['name_key'] = nba_names['name'].map(normalize_name)
nba_unique = nba_names[~nba_names.duplicated(['season', 'name_key'], keep=False)]

bb['name_key'] = bb['player'].map(normalize_name)
bb_unique = ~bb.duplicated(['season_type', 'season', 'name_key'], keep=False)
todo = bb['nba_player_id'].isna() & bb_unique
fallback = bb.loc[todo, ['season', 'name_key']].merge(nba_unique[['season', 'name_key', 'player_id']], on=['season', 'name_key'], how='left')
fallback.index = bb.index[todo]
bb.loc[todo, 'nba_player_id'] = fallback['player_id']
bb.loc[todo & bb['nba_player_id'].notna(), 'match'] = 'season + name'

print(bb['match'].value_counts(dropna=False).to_string())
bb.loc[bb['nba_player_id'].isna(), ['season_type', 'season', 'player', 'g', 'mp']].sort_values('mp', ascending=False).head(8)

match
bbref id         9794
season + name      20
NaN                 2


,season_type,season,player,g,mp
992,regular_season,2014,Vítor Luiz Faverani,37,488
1175,regular_season,2014,Tony Mitchell,3,10


### 4B. Check the matches against llimllib

If a Basketball-Reference row was attached to the right NBA player, both sources should report (nearly) the same number of games for that player and season.

In [12]:
check = bb.dropna(subset=['nba_player_id']).merge(
    stats_raw[['season', 'season_type', 'player_id', 'gp']].rename(columns={'player_id': 'nba_player_id'}),
    on=['season', 'season_type', 'nba_player_id'], how='left')
print('matched rows with a llimllib row for the same season and type:', check['gp'].notna().sum(), 'of', len(check))
same = (check['g'] == check['gp'])
print('...same games played:', f"{same.mean():.1%}", '| within 2 games:', f"{(check['g'] - check['gp']).abs().le(2).mean():.1%}")
print('\nby match type:')
print(check.assign(same=same).groupby('match')['same'].mean().round(3).to_string())
check.loc[(check['g'] - check['gp']).abs() > 2, ['season_type', 'season', 'player', 'g', 'gp', 'match']].head(8)

matched rows with a llimllib row for the same season and type: 9814 of 9814
...same games played: 100.0% | within 2 games: 100.0%

by match type:
match
bbref id         1.0
season + name    1.0


,season_type,season,player,g,gp,match


In [13]:
# Coverage the other way: how many of notebook 01's / 02's rows get a BPM?
rs01 = pd.read_csv(PROCESSED / 'llimllib_player_regular_season_features.csv', dtype={'nba_player_id': str})
po02 = pd.read_csv(PROCESSED / 'llimllib_player_playoff_features.csv', dtype={'nba_player_id': str})
for name, tbl, st in [('regular season (01)', rs01, 'regular_season'), ('playoffs (02)', po02, 'playoffs')]:
    got = tbl.merge(bb.loc[bb['season_type'] == st, ['season', 'nba_player_id']], on=['season', 'nba_player_id'], how='left', indicator=True)
    print(f"{name}: {(got['_merge'] == 'both').mean():.1%} of {len(tbl):,} rows have a Basketball-Reference row")

regular season (01): 100.0% of 5,968 rows have a Basketball-Reference row


playoffs (02): 100.0% of 2,405 rows have a Basketball-Reference row


**What we found:**

- **All but 2 Basketball-Reference rows get an NBA id:** 9,794 through the id table and 20 by name. The 2 unmatched rows are from 2013-14 (Vítor Faverani and Tony Mitchell), a history season that only feeds the 3-year averages, so dropping them only costs those players' 2013-14 BPM.
- **The matches are right:** every matched row reports **the same games played** in both sources, for id matches and name matches alike.
- **Coverage is complete:** every row in notebooks 01 and 02 has a Basketball-Reference row, so no player-season in our window is missing BPM. (The one 0-minute row from section 3C is in a history season.)

### 4C. 3-year Box +/- average and final columns

The 3-year average uses the regular-season BPM of the **three seasons before** each season, with the shared `prior_seasons_average()` helper and the same window as notebook 01's other 3-year averages. That is why we downloaded regular-season pages back to 2012-13.

**One difference: it is weighted by minutes.** BPM is a rate per 100 possessions, so a season of 2 minutes shouldn't count as much as a season of 2,000 (section 3D: one tiny sample is +240.8). The weighted average is `sum(bpm × minutes) / sum(minutes)` over the prior seasons. The other 3-year averages (points, rebounds, ...) are per-game numbers that can't blow up like this, so they stay plain averages.

In [14]:
matched = bb.dropna(subset=['nba_player_id'])
rs_bpm = matched[matched['season_type'] == 'regular_season'][['season', 'nba_player_id', 'bbref_id', 'bpm']]
po_bpm = matched[matched['season_type'] == 'playoffs'][['season', 'nba_player_id', 'bbref_id', 'bpm']]
assert not rs_bpm.duplicated(['season', 'nba_player_id']).any() and not po_bpm.duplicated(['season', 'nba_player_id']).any()

rs_bpm = rs_bpm.merge(matched[matched['season_type'] == 'regular_season'][['season', 'nba_player_id', 'mp']],
                      on=['season', 'nba_player_id'], validate='one_to_one')
rs_bpm['bpm_x_mp'] = rs_bpm['bpm'] * rs_bpm['mp']
rs_bpm['mp_with_bpm'] = rs_bpm['mp'].where(rs_bpm['bpm'].notna())
trend = prior_seasons_average(rs_bpm, 'nba_player_id', ['bpm_x_mp', 'mp_with_bpm'], n=3)
# mean(bpm x mp) / mean(mp) over the same seasons = sum(bpm x mp) / sum(mp)
trend['avg_bpm_last_3'] = trend['avg_bpm_x_mp_last_3'] / trend['avg_mp_with_bpm_last_3'].replace(0, np.nan)
trend = trend.rename(columns={'seasons_played_last_3': 'bpm_seasons_last_3'})[['nba_player_id', 'season', 'avg_bpm_last_3', 'bpm_seasons_last_3']]
rs_bpm = rs_bpm.drop(columns=['mp', 'bpm_x_mp', 'mp_with_bpm']).merge(trend, on=['nba_player_id', 'season'], how='left', validate='one_to_one')

bpm_features = (rs_bpm.rename(columns={'bpm': 'rs_bpm'})
                .merge(po_bpm.rename(columns={'bpm': 'po_bpm', 'bbref_id': 'bbref_id_po'}), on=['season', 'nba_player_id'], how='outer')
                .query('@FIRST_SEASON <= season <= @LAST_SEASON'))
bpm_features['bbref_id'] = bpm_features['bbref_id'].fillna(bpm_features['bbref_id_po'])
bpm_features = (bpm_features[['season', 'nba_player_id', 'bbref_id', 'rs_bpm', 'po_bpm', 'avg_bpm_last_3', 'bpm_seasons_last_3']]
                .sort_values(['season', 'nba_player_id']).reset_index(drop=True))
bpm_features['bpm_seasons_last_3'] = bpm_features['bpm_seasons_last_3'].fillna(0).astype(int)

jokic = bpm_features.merge(players[['player_id', 'name']], left_on='nba_player_id', right_on='player_id')
jokic[jokic['name'] == 'Nikola Jokić'][['season', 'rs_bpm', 'po_bpm', 'avg_bpm_last_3', 'bpm_seasons_last_3']]

,season,rs_bpm,po_bpm,avg_bpm_last_3,bpm_seasons_last_3
411,2016,3.8,NaN,NaN,0
916,2017,7.2,NaN,3.800000,1
1461,2018,6.9,NaN,5.637497,2
2011,2019,9.0,11.5,6.133843,3
2549,2020,7.4,7.3,7.740344,3
3090,2021,12.0,9.0,7.782384,3
3697,2022,13.6,10.6,9.508515,3
4245,2023,12.9,12.8,11.070685,3
4819,2024,13.1,12.7,12.830561,3
5388,2025,13.3,10.7,13.202627,3


## 5. Validate the final table

In [15]:
f = bpm_features
checks = {
    'one row per season + player': not f.duplicated(['season', 'nba_player_id']).any(),
    f'seasons are {FIRST_SEASON}-{LAST_SEASON}': f['season'].between(FIRST_SEASON, LAST_SEASON).all() and f['season'].nunique() == 11,
    'every row has an NBA id and a bbref id': f[['nba_player_id', 'bbref_id']].notna().all().all(),
    'average is blank whenever there were no prior seasons': f.loc[f['bpm_seasons_last_3'] == 0, 'avg_bpm_last_3'].isna().all(),
    'every regular-season row of notebook 01 has a row here': len(rs01.merge(f, on=['season', 'nba_player_id'])) == len(rs01),
    'the 3-year average stays within the single-season range': f['avg_bpm_last_3'].dropna().between(bb['bpm'].min(), bb['bpm'].max()).all(),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())
f.drop(columns='season').describe().T[['count', 'mean', 'min', '50%', 'max']].round(2)

PASS - one row per season + player
PASS - seasons are 2016-2026
PASS - every row has an NBA id and a bbref id
PASS - average is blank whenever there were no prior seasons
PASS - every regular-season row of notebook 01 has a row here
PASS - the 3-year average stays within the single-season range


,count,mean,min,50%,max
rs_bpm,5968.0,-1.54,-54.4,-1.30,240.8
po_bpm,2405.0,-0.61,-58.2,-0.10,70.1
avg_bpm_last_3,4822.0,-0.86,-33.9,-0.93,240.8
bpm_seasons_last_3,5975.0,1.98,0.0,3.00,3.0


**What we found:** Every check passes. There is one row per player-season, each with an NBA id. `po_bpm` is blank for players who didn't play in the playoffs, and `rs_bpm` only for the 7 playoff-only players. `avg_bpm_last_3` is blank for players with no Basketball-Reference season in the three previous seasons (mostly rookies). Its extreme values belong to players whose only prior season was a tiny sample; `seasons_played_last_3` and `avg_mpg_last_3` show that. Nikola Jokić's rows show the average working: blank as a rookie in 2015-16, then climbing from 3.8 to about 13 as his MVP seasons enter the window.

## 6. Save

In [16]:
out_path = PROCESSED / 'bbref_player_bpm_features.csv'
bpm_features.to_csv(out_path, index=False)
print('saved', bpm_features.shape, 'to', out_path)

saved (5975, 7) to ..\..\data-files\02-processed\bbref_player_bpm_features.csv


**Output:** `data-files/02-processed/bbref_player_bpm_features.csv`, **one row per player + season**. Keys: `season` + `nba_player_id` (`bbref_id` kept for reference). `bpm_seasons_last_3` is only used by notebook 999 to check against notebook 01's `seasons_played_last_3`.